<a href="https://colab.research.google.com/github/hbankar11/Pyspark_Learning/blob/main/CompletePySpark_2026_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
#Just Checking PySpark is Installed or not
from pyspark.sql import SparkSession

In [3]:
# Create spark session
spark_session = SparkSession.builder.appName("CompletePySpark_2026-04").getOrCreate()

In [ ]:
#Create SparkContext in PySpark
print(spark_session.sparkContext)

<SparkContext master=local[*] appName=CompletePySpark_2026-04>


In [ ]:
print("Spark App Name : " + spark_session.sparkContext.appName)

Spark App Name : CompletePySpark_2026-04


In [ ]:
#Stop PySpark SparkContext
#When PySpark executes this statement, it logs the message INFO SparkContext: Successfully stopped SparkContext to console or to a log file.
spark_session.sparkContext.stop()

In [ ]:
#Create RDD using parallelize
data = [1,2,3,4,5,6,7,8,9,10]
rdd = spark_session.sparkContext.parallelize(data)
rdd.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [ ]:
#create RDD using sparkContext.textFile()
rdd = spark_session.sparkContext.textFile("<path>")


In [ ]:
#create RDD using sparkContext.wholeTextFiles()
rdd3 = spark_session.sparkContext.wholeTextFiles("<path>")

In [ ]:
#Create empty RDD using sparkContext.emptyRDD
emptyRdd = spark_session.sparkContext.emptyRDD
print(emptyRdd)

<bound method SparkContext.emptyRDD of <SparkContext master=local[*] appName=CompletePySpark_2026-04>>


In [ ]:
#Creating empty RDD with partition
emptyRddPartition = spark_session.sparkContext.parallelize([],10)
emptyRddPartition.collect()

[]

In [ ]:
#RDD Partitions
#Get Partition count
print(f"No. of partition of rdd: ",rdd.getNumPartitions())

No. of partition of rdd:  2


In [ ]:
#Set partition Manually
rddPartition = spark_session.sparkContext.parallelize([1,2,3,4],4)
#rddPartition.collect()
rddPartition.getNumPartitions()

4

In [ ]:
#repartition and coalesce
rddPartition.repartition(6).getNumPartitions() #6
rddPartition.repartition(2).getNumPartitions() #2

rddPartition.coalesce(5).getNumPartitions() #4 Because coalesce is used only to decrease the partition
rddPartition.coalesce(2).getNumPartitions() #2

2

In [ ]:
#flatMap() : flattens all those sequences into a single RDD
rdd = spark_session.sparkContext.textFile("/content/sample.txt")
rdd.flatMap(lambda x:x.split(" ")).collect()

['1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '12',
 '23',
 '34',
 '45',
 '21',
 '27',
 '29',
 '30']

In [ ]:
#map() : It allows you to apply a function to each element of an RDD, producing a new RDD with the transformed elements.
# Add a new element with value 1 to each word
rdd_map = spark_session.sparkContext.textFile("/content/sample.txt").map(lambda x: (x,1))
rdd_map.collect()

[("('Project', 3)", 1),
 ("('Gutenberg’s', 3)", 1),
 ("('Alice’s', 1)", 1),
 ("('in', 2)", 1),
 ("('Adventures', 2)", 1),
 ("('Wonderland', 2)", 1)]

In [ ]:
#reduceByKey() : combines the values associated with each key using the provided function.
rdd_reduceByKey = spark_session.sparkContext.textFile("/content/sample.txt").reduceByKey(lambda a,b: a+b)


In [ ]:
#sortByKey() : transformation arranges the elements of an RDD based on their keys.
# Sample data: (key, value) pairs
data = [(3, "apple"), (1, "banana"), (4, "grapes"), (2, "orange")]

# Parallelize into an RDD
rdd = spark_session.sparkContext.parallelize(data)

# Sort by key
sorted_rdd = rdd.sortByKey()

# Collect results
print(sorted_rdd.collect())

[(1, 'banana'), (2, 'orange'), (3, 'apple'), (4, 'grapes')]


In [ ]:
data = [(3, "apple"), (1, "banana"), (4, "grapes"), (2, "orange")]
rdd = spark_session.sparkContext.parallelize(data)

# Parallelize into an RDD
rdd = spark_session.sparkContext.parallelize(data)
#1: count() => Returns the number of records in an RDD
print("Count : "+str(rdd.count()))
#2: first() => Returns the first record.
firstRec = rdd.first()
print("First Record : "+str(firstRec))
#3: max() => Returns maximum record
datMax = rdd.max()
print("Maximum Record : "+str(datMax))
#4: reduce() => Reduces the records to single, we can use this to count or sum.
totalWordCount = rdd.reduce(lambda a,b: (a[0]+b[0],a[1]))
print("dataReduce Record : "+str(totalWordCount[0]))
#5: take() =>  Returns the record specified as an argument.
data = rdd.take(2)
print("Element at position 2 : "+str(data))


Count : 4
First Record : (3, 'apple')
Maximum Record : (4, 'grapes')
dataReduce Record : 10
Element at position 3 : [(3, 'apple'), (1, 'banana')]


In [ ]:
## Converts RDD to DataFrame
dfRdd1 = rdd.toDF()
dfRdd1.show(truncate=False)

dfRdd1 = rdd.toDF(["id","fruit"])
dfRdd1.show(truncate=False)


+---+------+
|_1 |_2    |
+---+------+
|3  |apple |
|1  |banana|
|4  |grapes|
|2  |orange|
+---+------+

+---+------+
|id |fruit |
+---+------+
|3  |apple |
|1  |banana|
|4  |grapes|
|2  |orange|
+---+------+



In [ ]:
# using createDataFrame() - Convert DataFrame to RDD
df = spark_session.createDataFrame(rdd).toDF("id","fruit")
df.show(truncate=False)

+---+------+
|id |fruit |
+---+------+
|3  |apple |
|1  |banana|
|4  |grapes|
|2  |orange|
+---+------+



In [ ]:
# Convert DataFrame to RDD
rdd = df.rdd
rdd.collect()

[Row(id=3, fruit='apple'),
 Row(id=1, fruit='banana'),
 Row(id=4, fruit='grapes'),
 Row(id=2, fruit='orange')]

In [ ]:
#1: Create empty RDD in Pyspark
emptyRdd = spark_session.sparkContext.emptyRDD()
emptyRdd.collect()

[]

In [ ]:
#2: Alternative method to create empty RDD
emptyRdd1 = spark_session.sparkContext.parallelize([])
emptyRdd1.collect()

[]

In [ ]:
#3:Create Empty DataFrame with Schema (StructType)
from pyspark.sql.types import *
schema = StructType([
    StructField("id",StringType(),True),
    StructField("name",StringType(),True)
])
emptyRdd = spark_session.sparkContext.emptyRDD()
df = spark_session.createDataFrame(emptyRdd,schema)
df.show(truncate=False)

+---+----+
|id |name|
+---+----+
+---+----+



In [ ]:
#Convert Empty RDD to DataFrame
df = emptyRdd.toDF(schema)
df.show(truncate=False)

+---+----+
|id |name|
+---+----+
+---+----+



In [ ]:
#Create Empty DataFrame with Schema.
df = spark_session.createDataFrame([],schema)
df.show()

+---+----+
| id|name|
+---+----+
+---+----+



In [ ]:
#Create Empty DataFrame without Schema (no columns)
df = spark_session.createDataFrame([],StructType([]))
df.show()

++
||
++
++



In [ ]:
#Create pyspark RDD
dept = [("Finance",10),("Marketing",20),("Sales",30),("IT",40)]
rdd = spark_session.sparkContext.parallelize(dept)
rdd.collect()

[('Finance', 10), ('Marketing', 20), ('Sales', 30), ('IT', 40)]

In [ ]:
#Convert Pyspark rdd to dataframe
df = rdd.toDF()
df.show()

+---------+---+
|       _1| _2|
+---------+---+
|  Finance| 10|
|Marketing| 20|
|    Sales| 30|
|       IT| 40|
+---------+---+



In [ ]:
deptColumns = ["deptName","dept_id"]
df = rdd.toDF(deptColumns)
df.show(truncate=False)

+---------+-------+
|deptName |dept_id|
+---------+-------+
|Finance  |10     |
|Marketing|20     |
|Sales    |30     |
|IT       |40     |
+---------+-------+



In [ ]:
#Using Pyspark createDataFrame() function
df = spark_session.createDataFrame(rdd,deptColumns)
df.show(truncate=False)

+---------+-------+
|deptName |dept_id|
+---------+-------+
|Finance  |10     |
|Marketing|20     |
|Sales    |30     |
|IT       |40     |
+---------+-------+



In [ ]:
# Using createDataFrame() with StructType schema
deptSchema = StructType([
    StructField("dept_name",StringType(),True),
    StructField("dept_id",IntegerType(),True)
])

dept_df = spark_session.createDataFrame(rdd,deptSchema)
dept_df.show(truncate=False)


+---------+-------+
|dept_name|dept_id|
+---------+-------+
|Finance  |10     |
|Marketing|20     |
|Sales    |30     |
|IT       |40     |
+---------+-------+



In [ ]:
#Convert above pyspark dataframe to Pandas
#Step-1:
data = [("James","","Smith","36636","M",60000),
        ("Michael","Rose","","40288","M",70000),
        ("Robert","","Williams","42114","",400000),
        ("Maria","Anne","Jones","39192","F",500000),
        ("Jen","Mary","Brown","","F",0)]

columns = ["first_name","middle_name","last_name","dob","gender","salary"]

pyspark_df = spark_session.createDataFrame(data,columns)
pyspark_df.show(truncate=False)

#Step-2: Convert pyspark dataframe to pandas
pandas_df = pyspark_df.toPandas()
print(pandas_df)

+----------+-----------+---------+-----+------+------+
|first_name|middle_name|last_name|dob  |gender|salary|
+----------+-----------+---------+-----+------+------+
|James     |           |Smith    |36636|M     |60000 |
|Michael   |Rose       |         |40288|M     |70000 |
|Robert    |           |Williams |42114|      |400000|
|Maria     |Anne       |Jones    |39192|F     |500000|
|Jen       |Mary       |Brown    |     |F     |0     |
+----------+-----------+---------+-----+------+------+

  first_name middle_name last_name    dob gender  salary
0      James                 Smith  36636      M   60000
1    Michael        Rose            40288      M   70000
2     Robert              Williams  42114         400000
3      Maria        Anne     Jones  39192      F  500000
4        Jen        Mary     Brown             F       0


In [ ]:
#PySpark show() – Display DataFrame Contents in Table
#1:Default - displays 20 rows and 20 charactes from column value
columns = ["Seqno","Quote"]
data = [("1", "Be the change that you wish to see in the world"),
    ("2", "Everyone thinks of changing the world, but no one thinks of changing himself."),
    ("3", "The purpose of our lives is to be happy."),
    ("4", "Be cool.")]
df = spark_session.createDataFrame(data,columns)
df.show()

+-----+--------------------+
|Seqno|               Quote|
+-----+--------------------+
|    1|Be the change tha...|
|    2|Everyone thinks o...|
|    3|The purpose of ou...|
|    4|            Be cool.|
+-----+--------------------+



In [ ]:
#display full column contents
df.show(truncate=False)

+-----+-----------------------------------------------------------------------------+
|Seqno|Quote                                                                        |
+-----+-----------------------------------------------------------------------------+
|1    |Be the change that you wish to see in the world                              |
|2    |Everyone thinks of changing the world, but no one thinks of changing himself.|
|3    |The purpose of our lives is to be happy.                                     |
|4    |Be cool.                                                                     |
+-----+-----------------------------------------------------------------------------+



In [ ]:
# Display 2 rows and full column contents
df.show(2,truncate=False)

+-----+-----------------------------------------------------------------------------+
|Seqno|Quote                                                                        |
+-----+-----------------------------------------------------------------------------+
|1    |Be the change that you wish to see in the world                              |
|2    |Everyone thinks of changing the world, but no one thinks of changing himself.|
+-----+-----------------------------------------------------------------------------+
only showing top 2 rows


In [ ]:
# Display 2 rows & column values 25 characters
df.show(2,truncate=25)

+-----+-------------------------+
|Seqno|                    Quote|
+-----+-------------------------+
|    1|Be the change that you...|
|    2|Everyone thinks of cha...|
+-----+-------------------------+
only showing top 2 rows


In [ ]:
# Display DataFrame rows & columns vertically
df.show(n=3,truncate=25,vertical=True)

-RECORD 0--------------------------
 Seqno | 1                         
 Quote | Be the change that you... 
-RECORD 1--------------------------
 Seqno | 2                         
 Quote | Everyone thinks of cha... 
-RECORD 2--------------------------
 Seqno | 3                         
 Quote | The purpose of our liv... 
only showing top 3 rows


In [ ]:
#PySpark StructType & StructField Explained with Examples
#Using PySpark StructType & StructField with DataFrame
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
data = [("James","","Smith",36636,"M",3000),
    ("Michael","Rose","",40288,"M",4000),
    ("Robert","","Williams",42114,"M",4000),
    ("Maria","Anne","Jones",39192,"F",4000),
    ("Jen","Mary","Brown",None,"F",-1)
  ]

schema = StructType([
    StructField("First_Name",StringType(),True),
    StructField("Middle_Name",StringType(),True),
    StructField("Last_Name",StringType(),True),
    StructField("ID",IntegerType(),True),
    StructField("Gender",StringType(),True),
    StructField("Salary",StringType(),True)
])

df = spark_session.createDataFrame(data,schema)
df.show(truncate=False)


+----------+-----------+---------+-----+------+------+
|First_Name|Middle_Name|Last_Name|ID   |Gender|Salary|
+----------+-----------+---------+-----+------+------+
|James     |           |Smith    |36636|M     |3000  |
|Michael   |Rose       |         |40288|M     |4000  |
|Robert    |           |Williams |42114|M     |4000  |
|Maria     |Anne       |Jones    |39192|F     |4000  |
|Jen       |Mary       |Brown    |NULL |F     |-1    |
+----------+-----------+---------+-----+------+------+



In [ ]:
#nested structure:
from pyspark.sql.types import *
structureData = [
    (("James","","Smith"),"36636","M",3100),
    (("Michael","Rose",""),"40288","M",4300),
    (("Robert","","Williams"),"42114","M",1400),
    (("Maria","Anne","Jones"),"39192","F",5500),
    (("Jen","Mary","Brown"),"","F",-1)
  ]

nested_schema = StructType([
    StructField("name",StructType([
        StructField("first_name",StringType(),True),
        StructField("middle_name",StringType(),True),
        StructField("last_name",StringType(),True)
    ]),True),
    StructField("id",StringType(),True),
    StructField("gender",StringType(),True),
    StructField("salary",IntegerType(),True)
])

df = spark_session.createDataFrame(structureData,nested_schema)
df.show(truncate=False)
'''
+--------------------+-----+------+------+
|name                |id   |gender|salary|
+--------------------+-----+------+------+
|{James, , Smith}    |36636|M     |3100  |
|{Michael, Rose, }   |40288|M     |4300  |
|{Robert, , Williams}|42114|M     |1400  |
|{Maria, Anne, Jones}|39192|F     |5500  |
|{Jen, Mary, Brown}  |     |F     |-1    |
+--------------------+-----+------+------+
'''

+--------------------+-----+------+------+
|name                |id   |gender|salary|
+--------------------+-----+------+------+
|{James, , Smith}    |36636|M     |3100  |
|{Michael, Rose, }   |40288|M     |4300  |
|{Robert, , Williams}|42114|M     |1400  |
|{Maria, Anne, Jones}|39192|F     |5500  |
|{Jen, Mary, Brown}  |     |F     |-1    |
+--------------------+-----+------+------+



'\n+--------------------+-----+------+------+\n|name                |id   |gender|salary|\n+--------------------+-----+------+------+\n|{James, , Smith}    |36636|M     |3100  |\n|{Michael, Rose, }   |40288|M     |4300  |\n|{Robert, , Williams}|42114|M     |1400  |\n|{Maria, Anne, Jones}|39192|F     |5500  |\n|{Jen, Mary, Brown}  |     |F     |-1    |\n+--------------------+-----+------+------+\n'

In [ ]:
#Updating existing structtype using struct
from pyspark.sql.functions import *
updated_df = df.withColumn("OtherInfo",
                           struct(col("id").alias("identifier"),
                                  col("gender").alias("gender"),
                                  col("salary").alias("salary"),
                                  when(col("salary").cast(IntegerType()) < 2000,"low") \
                                 .when(col("salary").cast(IntegerType()) < 4000,"Medium") \
                                 .otherwise("High").alias("Salary_Grade"))).drop("id","gender","salary")

updated_df.show(truncate=False)
'''
+--------------------+------------------------+
|name                |OtherInfo               |
+--------------------+------------------------+
|{James, , Smith}    |{36636, M, 3100, Medium}|
|{Michael, Rose, }   |{40288, M, 4300, High}  |
|{Robert, , Williams}|{42114, M, 1400, low}   |
|{Maria, Anne, Jones}|{39192, F, 5500, High}  |
|{Jen, Mary, Brown}  |{, F, -1, low}          |
+--------------------+------------------------+
'''

+--------------------+------------------------+
|name                |OtherInfo               |
+--------------------+------------------------+
|{James, , Smith}    |{36636, M, 3100, Medium}|
|{Michael, Rose, }   |{40288, M, 4300, High}  |
|{Robert, , Williams}|{42114, M, 1400, low}   |
|{Maria, Anne, Jones}|{39192, F, 5500, High}  |
|{Jen, Mary, Brown}  |{, F, -1, low}          |
+--------------------+------------------------+



In [ ]:
arrayStructureSchema = StructType([
    StructField('name', StructType([
       StructField('firstname', StringType(), True),
       StructField('middlename', StringType(), True),
       StructField('lastname', StringType(), True)
       ])),
       StructField('hobbies', ArrayType(StringType()), True),
       StructField('properties', MapType(StringType(),StringType()), True)
    ])


In [ ]:
#Check if column exists in the dataframe

if 'firstname' in df.columns:
  print("firstname is present in the Dataframe")
else:
  print("firstname is not present in the Dataframe")

firstname is not present in the Dataframe


In [ ]:
column_name = 'id'
if column_name  in [field.name for field in df.schema]:
  print(f"'{column_name}' is present in the DataFrame schema.")
else:
    print(f"'{column_name}' is not present in the DataFrame schema.")

'id' is present in the DataFrame schema.


In [ ]:
#Create column class object:
from pyspark.sql.functions import lit
colObj = lit("HemanTejal")

In [ ]:
#Accessing columns from dataframe
data=[("James",23),("Ann",40)]
df=spark_session.createDataFrame(data).toDF("name.fname","gender")
df.printSchema()
'''
root
 |-- name.fname: string (nullable = true)
 |-- gender: long (nullable = true)
'''


root
 |-- name.fname: string (nullable = true)
 |-- gender: long (nullable = true)



In [ ]:
#1:# Using DataFrame object (df)
df.select(df.gender).show(truncate=False)
df.select(df["gender"],df["`name.fname`"]).show(truncate=False)
#2: Using col() function
df.select(col("gender"),col("`name.fname`")).show(truncate=False)
df.select(col("gender")).show(truncate=False)


+------+
|gender|
+------+
|23    |
|40    |
+------+

+------+----------+
|gender|name.fname|
+------+----------+
|23    |James     |
|40    |Ann       |
+------+----------+



In [ ]:
#Select all Columns
df.select(col("*")).show(truncate=False)
df.select("*").show(truncate=False)

+----------+------+
|name.fname|gender|
+----------+------+
|James     |23    |
|Ann       |40    |
+----------+------+



In [ ]:
#Create DataFrame with struct using Row class
from pyspark.sql import Row
from pyspark.sql.functions import col
data=[Row(name="James",prop=Row(hair="black",eye="blue")),
      Row(name="Ann",prop=Row(hair="grey",eye="black"))]

df = spark_session.createDataFrame(data)
df.printSchema()
'''
root
 |-- name: string (nullable = true)
 |-- prop: struct (nullable = true)
 |    |-- hair: string (nullable = true)
 |    |-- eye: string (nullable = true)
'''
#Access struct column:
df.select(df.prop.hair).show(truncate=False)
'''
+---------+
|prop.hair|
+---------+
|black    |
|grey     |
+---------+
'''
df.select(df["prop.eye"]).show(truncate=False)
'''
+-----+
|eye  |
+-----+
|blue |
|black|
+-----+
'''
#Access all columns from struct
df.select(col("prop.*")).show(truncate=False)
'''
+-----+-----+
|hair |eye  |
+-----+-----+
|black|blue |
|grey |black|
+-----+-----+
'''

root
 |-- name: string (nullable = true)
 |-- prop: struct (nullable = true)
 |    |-- hair: string (nullable = true)
 |    |-- eye: string (nullable = true)

+---------+
|prop.hair|
+---------+
|black    |
|grey     |
+---------+

+-----+
|eye  |
+-----+
|blue |
|black|
+-----+

+-----+-----+
|hair |eye  |
+-----+-----+
|black|blue |
|grey |black|
+-----+-----+



In [ ]:
#PySpark Column Operators
data=[(100,2,1),(200,3,4),(300,4,4)]
df=spark_session.createDataFrame(data).toDF("col1","col2","col3")

#Arithmetic operations
df.select(col("col1") + col("col2")).show(truncate=False)
df.select(col("col1") - col("col2")).show(truncate=False)
df.select(col("col1") * col("col2")).show(truncate=False)
df.select(col("col1") / col("col2")).show(truncate=False)
df.select(col("col1") % col("col2")).show(truncate=False)

df.select(col("col2") > col("col3")).show(truncate=False)
df.select(col("col2") < col("col3")).show(truncate=False)
'''
+-------------+
|(col1 + col2)|
+-------------+
|102          |
|203          |
|304          |
+-------------+

+-------------+
|(col1 - col2)|
+-------------+
|98           |
|197          |
|296          |
+-------------+

+-------------+
|(col1 * col2)|
+-------------+
|200          |
|600          |
|1200         |
+-------------+

+-----------------+
|(col1 / col2)    |
+-----------------+
|50.0             |
|66.66666666666667|
|75.0             |
+-----------------+

+-------------+
|(col1 % col2)|
+-------------+
|0            |
|2            |
|0            |
+-------------+

+-------------+
|(col2 > col3)|
+-------------+
|true         |
|false        |
|false        |
+-------------+

+-------------+
|(col2 < col3)|
+-------------+
|false        |
|true         |
|false        |
+-------------+
'''

+-------------+
|(col1 + col2)|
+-------------+
|102          |
|203          |
|304          |
+-------------+

+-------------+
|(col1 - col2)|
+-------------+
|98           |
|197          |
|296          |
+-------------+

+-------------+
|(col1 * col2)|
+-------------+
|200          |
|600          |
|1200         |
+-------------+

+-----------------+
|(col1 / col2)    |
+-----------------+
|50.0             |
|66.66666666666667|
|75.0             |
+-----------------+

+-------------+
|(col1 % col2)|
+-------------+
|0            |
|2            |
|0            |
+-------------+

+-------------+
|(col2 > col3)|
+-------------+
|true         |
|false        |
|false        |
+-------------+

+-------------+
|(col2 < col3)|
+-------------+
|false        |
|true         |
|false        |
+-------------+



In [ ]:
# PySpark Column Functions Example
data=[("James","Bond","100",None),
      ("Ann","Varsa","200",'F'),
      ("Tom Cruise","XXX","400",''),
      ("Tom Brand",None,"400",'M')]
columns=["fname","lname","id","gender"]
df=spark_session.createDataFrame(data,columns)
df.show(truncate=False)
'''
+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|James     |Bond |100|NULL  |
|Ann       |Varsa|200|F     |
|Tom Cruise|XXX  |400|      |
|Tom Brand |NULL |400|M     |
+----------+-----+---+------+
'''
#1: alias(): Set's name to column
df.select(df.fname.alias("first_name")).show(truncate=False)
'''
+----------+
|first_name|
+----------+
|James     |
|Ann       |
|Tom Cruise|
|Tom Brand |
+----------+
'''
#2: expr()
from pyspark.sql.functions import expr
df.select(expr(" fname || ',' || lname").alias("Full_Name")).show(truncate=False)
'''
+--------------+
|Full_Name     |
+--------------+
|James,Bond    |
|Ann,Varsa     |
|Tom Cruise,XXX|
|NULL          |
+--------------+
'''


+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|James     |Bond |100|NULL  |
|Ann       |Varsa|200|F     |
|Tom Cruise|XXX  |400|      |
|Tom Brand |NULL |400|M     |
+----------+-----+---+------+

+----------+
|first_name|
+----------+
|James     |
|Ann       |
|Tom Cruise|
|Tom Brand |
+----------+

+--------------+
|Full_Name     |
+--------------+
|James,Bond    |
|Ann,Varsa     |
|Tom Cruise,XXX|
|NULL          |
+--------------+



In [ ]:
#Sorting Technique
# #asc, desc to sort ascending and descending order repsectively.
df.sort(df.fname.asc()).show(truncate=False)
'''
+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|Ann       |Varsa|200|F     |
|James     |Bond |100|NULL  |
|Tom Brand |NULL |400|M     |
|Tom Cruise|XXX  |400|      |
+----------+-----+---+------+
'''
df.sort(df.lname.desc()).show(truncate=False)
'''
+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|Tom Cruise|XXX  |400|      |
|Ann       |Varsa|200|F     |
|James     |Bond |100|NULL  |
|Tom Brand |NULL |400|M     |
+----------+-----+---+------+
'''


+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|Ann       |Varsa|200|F     |
|James     |Bond |100|NULL  |
|Tom Brand |NULL |400|M     |
|Tom Cruise|XXX  |400|      |
+----------+-----+---+------+

+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|Tom Cruise|XXX  |400|      |
|Ann       |Varsa|200|F     |
|James     |Bond |100|NULL  |
|Tom Brand |NULL |400|M     |
+----------+-----+---+------+



In [ ]:
# Data Conversion in DataFrame
# Cast() and astype() is used to convert data type
df.select(df.id.astype("int")).printSchema()
df.select(df["id"].cast("int")).printSchema()

root
 |-- id: integer (nullable = true)

root
 |-- id: integer (nullable = true)



In [ ]:
# Between:
# Returns Boolean expression when column values in between lower and upper bound.
df.filter(df.id.between(100,200)).show(truncate=False)
'''
+-----+-----+---+------+
|fname|lname|id |gender|
+-----+-----+---+------+
|James|Bond |100|NULL  |
|Ann  |Varsa|200|F     |
+-----+-----+---+------+
'''

+-----+-----+---+------+
|fname|lname|id |gender|
+-----+-----+---+------+
|James|Bond |100|NULL  |
|Ann  |Varsa|200|F     |
+-----+-----+---+------+



In [ ]:
# contains: Checks if a DataFrame column value contains a value specified in this function.
df.filter(df.fname.contains("Cruise")).show(truncate=False)
'''
+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|Tom Cruise|XXX  |400|      |
+----------+-----+---+------+
'''

+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|Tom Cruise|XXX  |400|      |
+----------+-----+---+------+



In [ ]:
# startswith() and endswith() - Checks if the value of DataFrame starts and ends with string respectively.
df.filter(df['fname'].startswith("T")).show(truncate=False)
'''
+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|Tom Cruise|XXX  |400|      |
|Tom Brand |NULL |400|M     |
+----------+-----+---+------+
'''
df.filter(df.fname.endswith("e")).show(truncate=False)
'''
+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|Tom Cruise|XXX  |400|      |
+----------+-----+---+------+
'''

+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|Tom Cruise|XXX  |400|      |
|Tom Brand |NULL |400|M     |
+----------+-----+---+------+

+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|Tom Cruise|XXX  |400|      |
+----------+-----+---+------+



In [ ]:
# isNull and isNotNull
df.filter(df.lname.isNull()).show(truncate=False)
df.filter(df.lname.isNotNull()).show(truncate=False)
'''
+---------+-----+---+------+
|fname    |lname|id |gender|
+---------+-----+---+------+
|Tom Brand|NULL |400|M     |
+---------+-----+---+------+

+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|James     |Bond |100|NULL  |
|Ann       |Varsa|200|F     |
|Tom Cruise|XXX  |400|      |
+----------+-----+---+------+
'''

+---------+-----+---+------+
|fname    |lname|id |gender|
+---------+-----+---+------+
|Tom Brand|NULL |400|M     |
+---------+-----+---+------+

+----------+-----+---+------+
|fname     |lname|id |gender|
+----------+-----+---+------+
|James     |Bond |100|NULL  |
|Ann       |Varsa|200|F     |
|Tom Cruise|XXX  |400|      |
+----------+-----+---+------+



In [ ]:
# like() and rlike():
df.select("id","fname","lname").filter(df.fname.like("%om%")).show(truncate=False)
'''
+---+----------+-----+
|id |fname     |lname|
+---+----------+-----+
|400|Tom Cruise|XXX  |
|400|Tom Brand |NULL |
+---+----------+-----+
'''

+---+----------+-----+
|id |fname     |lname|
+---+----------+-----+
|400|Tom Cruise|XXX  |
|400|Tom Brand |NULL |
+---+----------+-----+



In [ ]:
#substr(): Returns a column after getting substring from the column.
df.select(df.fname.substr(1,2)).show(truncate=False)
'''
+-------------------+
|substr(fname, 1, 2)|
+-------------------+
|Ja                 |
|An                 |
|To                 |
|To                 |
+-------------------+
'''

+-------------------+
|substr(fname, 1, 2)|
+-------------------+
|Ja                 |
|An                 |
|To                 |
|To                 |
+-------------------+



'\n+-------------------+\n|substr(fname, 1, 2)|\n+-------------------+\n|Ja                 |\n|An                 |\n|To                 |\n|To                 |\n+-------------------+\n'

In [ ]:
#when() and otherwise():
'''
Input Dataframe:
+----------+-----+---+------+
|     fname|lname| id|gender|
+----------+-----+---+------+
|     James| Bond|100|  NULL|
|       Ann|Varsa|200|     F|
|Tom Cruise|  XXX|400|      |
| Tom Brand| NULL|400|     M|
+----------+-----+---+------+
'''
from pyspark.sql.functions import when
df.select(df.fname,df.lname,when(df.gender == 'M',"Male").when(df.gender == 'F',"Female").otherwise("N/A").alias("gender")).show(truncate=False)

+----------+-----+------+
|fname     |lname|gender|
+----------+-----+------+
|James     |Bond |N/A   |
|Ann       |Varsa|Female|
|Tom Cruise|XXX  |N/A   |
|Tom Brand |NULL |Male  |
+----------+-----+------+



In [ ]:
#isin() : Check if value presents in the list
li = [100,200]
df.select(df.fname,df.lname,df.id) \
  .filter(df.id.isin(li)).show(truncate=False)

'''
+-----+-----+---+
|fname|lname|id |
+-----+-----+---+
|James|Bond |100|
|Ann  |Varsa|200|
+-----+-----+---+
...

+-----+-----+---+
|fname|lname|id |
+-----+-----+---+
|James|Bond |100|
|Ann  |Varsa|200|
+-----+-----+---+



In [ ]:
#getField(): To get the value by key from MapType column and by struct child name from StructType column.
from pyspark.sql.types import *
data=[(("James","Bond"),["Java","C#"],{'hair':'black','eye':'brown'}),
      (("Ann","Varsa"),[".NET","Python"],{'hair':'brown','eye':'black'}),
      (("Tom Cruise",""),["Python","Scala"],{'hair':'red','eye':'grey'}),
      (("Tom Brand",None),["Perl","Ruby"],{'hair':'black','eye':'blue'})]

schema = StructType([
        StructField('name', StructType([
            StructField('fname', StringType(), True),
            StructField('lname', StringType(), True)])),
        StructField('languages', ArrayType(StringType()),True),
        StructField('properties', MapType(StringType(),StringType()),True)
     ])

df = spark_session.createDataFrame(data,schema)
df.show(truncate=False)
''''
+-----------------+---------------+-----------------------------+
|name             |languages      |properties                   |
+-----------------+---------------+-----------------------------+
|{James, Bond}    |[Java, C#]     |{eye -> brown, hair -> black}|
|{Ann, Varsa}     |[.NET, Python] |{eye -> black, hair -> brown}|
|{Tom Cruise, }   |[Python, Scala]|{eye -> grey, hair -> red}   |
|{Tom Brand, NULL}|[Perl, Ruby]   |{eye -> blue, hair -> black} |
+-----------------+---------------+-----------------------------+
'''


+-----------------+---------------+-----------------------------+
|name             |languages      |properties                   |
+-----------------+---------------+-----------------------------+
|{James, Bond}    |[Java, C#]     |{eye -> brown, hair -> black}|
|{Ann, Varsa}     |[.NET, Python] |{eye -> black, hair -> brown}|
|{Tom Cruise, }   |[Python, Scala]|{eye -> grey, hair -> red}   |
|{Tom Brand, NULL}|[Perl, Ruby]   |{eye -> blue, hair -> black} |
+-----------------+---------------+-----------------------------+

+----------------+
|properties[hair]|
+----------------+
|black           |
|brown           |
|red             |
|black           |
+----------------+



In [ ]:
#1: #getField from MapType
df.select(df.properties.getField("hair")).show(truncate=False)
'''
+----------------+
|properties[hair]|
+----------------+
|black           |
|brown           |
|red             |
|black           |
+----------------+
'''

+----------------+
|properties[hair]|
+----------------+
|black           |
|brown           |
|red             |
|black           |
+----------------+



In [ ]:
# Get feild from Struct
df.select(df.name.getField("fname")).show(truncate=False)
'''
+----------+
|name.fname|
+----------+
|James     |
|Ann       |
|Tom Cruise|
|Tom Brand |
+----------+
'''


+----------+
|name.fname|
+----------+
|James     |
|Ann       |
|Tom Cruise|
|Tom Brand |
+----------+



In [ ]:
#getItem(): To get the value by index from Map Type or Array Type and any key for MapType column
#getItem() used with ArrayType
df.select(df.languages.getItem(1)).show(truncate=False)
'''+------------+
|languages[1]|
+------------+
|C#          |
|Python      |
|Scala       |
|Ruby        |
+------------+
'''

+------------+
|languages[1]|
+------------+
|C#          |
|Python      |
|Scala       |
|Ruby        |
+------------+



In [ ]:
#getItem() used with MapType
df.select(df.properties.getItem("hair")).show()
'''
+----------------+
|properties[hair]|
+----------------+
|           black|
|           brown|
|             red|
|           black|
+----------------+
'''

+----------------+
|properties[hair]|
+----------------+
|           black|
|           brown|
|             red|
|           black|
+----------------+



In [ ]:
# withField () - PySpark Column's withField() method is used to either add or update a nested field value.
import pyspark.sql.functions as f

data = [
    Row(name="Alex", age=20, friend=Row(name="Bob",age=30)),
    Row(name="Cathy", age=40, friend=Row(name="Doge",age=40))
]

df = spark_session.createDataFrame(data)
df.show(truncate=False)
'''
+-----+---+----------+
|name |age|friend    |
+-----+---+----------+
|Alex |20 |{Bob, 30} |
|Cathy|40 |{Doge, 40}|
+-----+---+----------+
'''
#Updating value in nested column in field
updated_col = df["friend"].withField("name",f.lit("Hemant"))
df.withColumn("friend",updated_col).show(truncate=False)
'''
+-----+---+------------+
|name |age|friend      |
+-----+---+------------+
|Alex |20 |{Hemant, 30}|
|Cathy|40 |{Hemant, 40}|
+-----+---+------------+
'''

#Adding new field values in nested rows in pyspark
updated_col = df["friend"].withField("upper_name", f.upper("friend.name"))
df_new = df.withColumn("friend", updated_col)
df_new.show()
'''
+-----+---+----------+
|name |age|friend    |
+-----+---+----------+
|Alex |20 |{Bob, 30} |
|Cathy|40 |{Doge, 40}|
+-----+---+----------+

+-----+---+------------+
|name |age|friend      |
+-----+---+------------+
|Alex |20 |{Hemant, 30}|
|Cathy|40 |{Hemant, 40}|
+-----+---+------------+

+-----+---+----------------+
| name|age|          friend|
+-----+---+----------------+
| Alex| 20|  {Bob, 30, BOB}|
|Cathy| 40|{Doge, 40, DOGE}|
+-----+---+----------------+
'''

+-----+---+----------+
|name |age|friend    |
+-----+---+----------+
|Alex |20 |{Bob, 30} |
|Cathy|40 |{Doge, 40}|
+-----+---+----------+

+-----+---+------------+
|name |age|friend      |
+-----+---+------------+
|Alex |20 |{Hemant, 30}|
|Cathy|40 |{Hemant, 40}|
+-----+---+------------+

+-----+---+----------------+
| name|age|          friend|
+-----+---+----------------+
| Alex| 20|  {Bob, 30, BOB}|
|Cathy| 40|{Doge, 40, DOGE}|
+-----+---+----------------+



In [ ]:
#PySpark-Select ()
#Create a dataframe
data = [("James","Smith","USA","CA"),
    ("Michael","Rose","USA","NY"),
    ("Robert","Williams","USA","CA"),
    ("Maria","Jones","USA","FL")
]
columns = ["firstname","lastname","country","state"]

df = spark_session.createDataFrame(data,columns)
# Select single and multiple columns in pyspark
df.select("firstname","lastname").show(truncate=False)
'''
+---------+--------+
|firstname|lastname|
+---------+--------+
|James    |Smith   |
|Michael  |Rose    |
|Robert   |Williams|
|Maria    |Jones   |
+---------+--------+
'''


+---------+--------+
|firstname|lastname|
+---------+--------+
|James    |Smith   |
|Michael  |Rose    |
|Robert   |Williams|
|Maria    |Jones   |
+---------+--------+



In [ ]:
df.select(df.firstname,df.lastname).show(truncate=False)
'''
+---------+--------+
|firstname|lastname|
+---------+--------+
|James    |Smith   |
|Michael  |Rose    |
|Robert   |Williams|
|Maria    |Jones   |
+---------+--------+
'''

+---------+--------+
|firstname|lastname|
+---------+--------+
|James    |Smith   |
|Michael  |Rose    |
|Robert   |Williams|
|Maria    |Jones   |
+---------+--------+



In [ ]:
df.select(df["firstname"],df["lastname"]).show(truncate=False)
'''
+---------+--------+
|firstname|lastname|
+---------+--------+
|James    |Smith   |
|Michael  |Rose    |
|Robert   |Williams|
|Maria    |Jones   |
+---------+--------+
'''

+---------+--------+
|firstname|lastname|
+---------+--------+
|James    |Smith   |
|Michael  |Rose    |
|Robert   |Williams|
|Maria    |Jones   |
+---------+--------+



In [ ]:
# By using col() function
from pyspark.sql.functions import col
df.select(col("firstname"),col("lastname")).show(truncate=False)
'''
+---------+--------+
|firstname|lastname|
+---------+--------+
|James    |Smith   |
|Michael  |Rose    |
|Robert   |Williams|
|Maria    |Jones   |
+---------+--------+
'''

+---------+--------+
|firstname|lastname|
+---------+--------+
|James    |Smith   |
|Michael  |Rose    |
|Robert   |Williams|
|Maria    |Jones   |
+---------+--------+



In [ ]:
# Select column by regular expression
df.select(df.colRegex("`^.*name*`")).show()

In [ ]:
# # Select All columns from List
df.select(*columns).show(truncate=False)
'''
+---------+--------+-------+-----+
|firstname|lastname|country|state|
+---------+--------+-------+-----+
|James    |Smith   |USA    |CA   |
|Michael  |Rose    |USA    |NY   |
|Robert   |Williams|USA    |CA   |
|Maria    |Jones   |USA    |FL   |
+---------+--------+-------+-----+
'''

+---------+--------+-------+-----+
|firstname|lastname|country|state|
+---------+--------+-------+-----+
|James    |Smith   |USA    |CA   |
|Michael  |Rose    |USA    |NY   |
|Robert   |Williams|USA    |CA   |
|Maria    |Jones   |USA    |FL   |
+---------+--------+-------+-----+



In [ ]:
## Select All columns
df.select([col for col in df.columns]).show(truncate=False)
df.select("*").show()
'''
+---------+--------+-------+-----+
|firstname|lastname|country|state|
+---------+--------+-------+-----+
|James    |Smith   |USA    |CA   |
|Michael  |Rose    |USA    |NY   |
|Robert   |Williams|USA    |CA   |
|Maria    |Jones   |USA    |FL   |
+---------+--------+-------+-----+
'''

+---------+--------+-------+-----+
|firstname|lastname|country|state|
+---------+--------+-------+-----+
|James    |Smith   |USA    |CA   |
|Michael  |Rose    |USA    |NY   |
|Robert   |Williams|USA    |CA   |
|Maria    |Jones   |USA    |FL   |
+---------+--------+-------+-----+



In [ ]:
# Select columns by index
#1: Selects first 3 columns and top 3 rows
df.select(df.columns[:3]).show(3,truncate=False)
'''
+---------+--------+-------+
|firstname|lastname|country|
+---------+--------+-------+
|James    |Smith   |USA    |
|Michael  |Rose    |USA    |
|Robert   |Williams|USA    |
+---------+--------+-------+
only showing top 3 rows
'''

+---------+--------+-------+
|firstname|lastname|country|
+---------+--------+-------+
|James    |Smith   |USA    |
|Michael  |Rose    |USA    |
|Robert   |Williams|USA    |
+---------+--------+-------+
only showing top 3 rows


In [ ]:
# #Selects columns 2 to 4  and top 3 rows
df.select(df.columns[2:4]).show(3,truncate=False)
'''
+-------+-----+
|country|state|
+-------+-----+
|USA    |CA   |
|USA    |NY   |
|USA    |CA   |
+-------+-----+
only showing top 3 rows
'''

+-------+-----+
|country|state|
+-------+-----+
|USA    |CA   |
|USA    |NY   |
|USA    |CA   |
+-------+-----+
only showing top 3 rows


In [8]:
from pyspark.sql import types
# Select Nested Struct Columns from PySpark
data = [
        (("James",None,"Smith"),"OH","M"),
        (("Anna","Rose",""),"NY","F"),
        (("Julia","","Williams"),"OH","F"),
        (("Maria","Anne","Jones"),"NY","M"),
        (("Jen","Mary","Brown"),"NY","M"),
        (("Mike","Mary","Williams"),"OH","M")
]

from pyspark.sql.types import *

schema = StructType([
    StructField('name',StructType([
        StructField("firstname",StringType(),True),
        StructField("middlename",StringType(),True),
        StructField("lastname",StringType(),True)
    ])),
    StructField("state",StringType(),True),
    StructField("gender",StringType(),True)
])

df = spark_session.createDataFrame(data,schema)
df.show(truncate=False)
'''
+----------------------+-----+------+
|name                  |state|gender|
+----------------------+-----+------+
|{James, NULL, Smith}  |OH   |M     |
|{Anna, Rose, }        |NY   |F     |
|{Julia, , Williams}   |OH   |F     |
|{Maria, Anne, Jones}  |NY   |M     |
|{Jen, Mary, Brown}    |NY   |M     |
|{Mike, Mary, Williams}|OH   |M     |
+----------------------+-----+------+
'''


+----------------------+-----+------+
|name                  |state|gender|
+----------------------+-----+------+
|{James, NULL, Smith}  |OH   |M     |
|{Anna, Rose, }        |NY   |F     |
|{Julia, , Williams}   |OH   |F     |
|{Maria, Anne, Jones}  |NY   |M     |
|{Jen, Mary, Brown}    |NY   |M     |
|{Mike, Mary, Williams}|OH   |M     |
+----------------------+-----+------+



'\n+----------------------+-----+------+\n|name                  |state|gender|\n+----------------------+-----+------+\n|{James, NULL, Smith}  |OH   |M     |\n|{Anna, Rose, }        |NY   |F     |\n|{Julia, , Williams}   |OH   |F     |\n|{Maria, Anne, Jones}  |NY   |M     |\n|{Jen, Mary, Brown}    |NY   |M     |\n|{Mike, Mary, Williams}|OH   |M     |\n+----------------------+-----+------+\n'

In [5]:
#1: Select Struct column i.e. name
df.select("name").show(truncate=False)
'''
+----------------------+
|name                  |
+----------------------+
|{James, NULL, Smith}  |
|{Anna, Rose, }        |
|{Julia, , Williams}   |
|{Maria, Anne, Jones}  |
|{Jen, Mary, Brown}    |
|{Mike, Mary, Williams}|
+----------------------+
'''

+----------------------+
|name                  |
+----------------------+
|{James, NULL, Smith}  |
|{Anna, Rose, }        |
|{Julia, , Williams}   |
|{Maria, Anne, Jones}  |
|{Jen, Mary, Brown}    |
|{Mike, Mary, Williams}|
+----------------------+



In [10]:
#2: Select nested struct column
df.printSchema()
'''
root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- state: string (nullable = true)
 |-- gender: string (nullable = true)
'''
df.select("name.firstname","name.middlename","name.lastname").show(truncate=False)
'''
+---------+----------+--------+
|firstname|middlename|lastname|
+---------+----------+--------+
|James    |NULL      |Smith   |
|Anna     |Rose      |        |
|Julia    |          |Williams|
|Maria    |Anne      |Jones   |
|Jen      |Mary      |Brown   |
|Mike     |Mary      |Williams|
+---------+----------+--------+
'''

root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- state: string (nullable = true)
 |-- gender: string (nullable = true)

+---------+----------+--------+
|firstname|middlename|lastname|
+---------+----------+--------+
|James    |NULL      |Smith   |
|Anna     |Rose      |        |
|Julia    |          |Williams|
|Maria    |Anne      |Jones   |
|Jen      |Mary      |Brown   |
|Mike     |Mary      |Williams|
+---------+----------+--------+



In [11]:
#3: Display all columns from struct column
df.select("name.*").show(truncate=False)
'''
+---------+----------+--------+
|firstname|middlename|lastname|
+---------+----------+--------+
|James    |NULL      |Smith   |
|Anna     |Rose      |        |
|Julia    |          |Williams|
|Maria    |Anne      |Jones   |
|Jen      |Mary      |Brown   |
|Mike     |Mary      |Williams|
+---------+----------+--------+
'''

+---------+----------+--------+
|firstname|middlename|lastname|
+---------+----------+--------+
|James    |NULL      |Smith   |
|Anna     |Rose      |        |
|Julia    |          |Williams|
|Maria    |Anne      |Jones   |
|Jen      |Mary      |Brown   |
|Mike     |Mary      |Williams|
+---------+----------+--------+



In [14]:
#PySpark - Collect() - PySpark RDD/DataFrame collect () is an action operation that is used to retrieve all the elements of the dataset (from all nodes) to the driver node.
# Limitation of collect() - retrieving larger datasets results in OutOfMemory error

# Step - 1: Let's use show() method
dept = [("Finance",10),("Marketing",20),("Sales",30),("IT",40)]
deptColumns = ["Dept_Name","Dept_id"]
deptDF = spark_session.createDataFrame(data = dept, schema = deptColumns)
deptDF.show(truncate = False)
'''
+---------+-------+
|Dept_Name|Dept_id|
+---------+-------+
|Finance  |10     |
|Marketing|20     |
|Sales    |30     |
|IT       |40     |
+---------+-------+
'''
# Now let's use collect()
dataCollect = df.collect()
print(dataCollect) #returns data in an Array to the driver
# [Row(name=Row(firstname='James', middlename=None, lastname='Smith'), state='OH', gender='M'), Row(name=Row(firstname='Anna', middlename='Rose', lastname=''), state='NY', gender='F'), Row(name=Row(firstname='Julia', middlename='', lastname='Williams'), state='OH', gender='F'), Row(name=Row(firstname='Maria', middlename='Anne', lastname='Jones'), state='NY', gender='M'), Row(name=Row(firstname='Jen', middlename='Mary', lastname='Brown'), state='NY', gender='M'), Row(name=Row(firstname='Mike', middlename='Mary', lastname='Williams'), state='OH', gender='M')]


+---------+-------+
|Dept_Name|Dept_id|
+---------+-------+
|Finance  |10     |
|Marketing|20     |
|Sales    |30     |
|IT       |40     |
+---------+-------+

[Row(name=Row(firstname='James', middlename=None, lastname='Smith'), state='OH', gender='M'), Row(name=Row(firstname='Anna', middlename='Rose', lastname=''), state='NY', gender='F'), Row(name=Row(firstname='Julia', middlename='', lastname='Williams'), state='OH', gender='F'), Row(name=Row(firstname='Maria', middlename='Anne', lastname='Jones'), state='NY', gender='M'), Row(name=Row(firstname='Jen', middlename='Mary', lastname='Brown'), state='NY', gender='M'), Row(name=Row(firstname='Mike', middlename='Mary', lastname='Williams'), state='OH', gender='M')]


In [15]:
for row in dataCollect:
  print(row)
'''
Row(name=Row(firstname='James', middlename=None, lastname='Smith'), state='OH', gender='M')
Row(name=Row(firstname='Anna', middlename='Rose', lastname=''), state='NY', gender='F')
Row(name=Row(firstname='Julia', middlename='', lastname='Williams'), state='OH', gender='F')
Row(name=Row(firstname='Maria', middlename='Anne', lastname='Jones'), state='NY', gender='M')
Row(name=Row(firstname='Jen', middlename='Mary', lastname='Brown'), state='NY', gender='M')
Row(name=Row(firstname='Mike', middlename='Mary', lastname='Williams'), state='OH', gender='M')
'''


Row(name=Row(firstname='James', middlename=None, lastname='Smith'), state='OH', gender='M')
Row(name=Row(firstname='Anna', middlename='Rose', lastname=''), state='NY', gender='F')
Row(name=Row(firstname='Julia', middlename='', lastname='Williams'), state='OH', gender='F')
Row(name=Row(firstname='Maria', middlename='Anne', lastname='Jones'), state='NY', gender='M')
Row(name=Row(firstname='Jen', middlename='Mary', lastname='Brown'), state='NY', gender='M')
Row(name=Row(firstname='Mike', middlename='Mary', lastname='Williams'), state='OH', gender='M')


In [19]:
#Returns value of First Row, First Column which is "Finance"
deptDF.collect()[0]
#Row(Dept_Name='Finance', Dept_id=10)

deptDF.collect()[0][0]
#Finance

'Finance'

In [22]:
#PySpark - withColumn()
data = [('James','','Smith','1991-04-01','M',3000),
        ('Michael','Rose','','2000-05-19','M',4000),
        ('Robert','','Williams','1978-09-05','M',4000),
        ('Maria','Anne','Jones','1967-12-01','F',4000),
        ('Jen','Mary','Brown','1980-02-17','F',-1)
]

columns = ["firstname","middlename","lastname","dob","gender","salary"]

df = spark_session.createDataFrame(data,columns)
df.show(truncate=False)
'''
+---------+----------+--------+----------+------+------+
|firstname|middlename|lastname|dob       |gender|salary|
+---------+----------+--------+----------+------+------+
|James    |          |Smith   |1991-04-01|M     |3000  |
|Michael  |Rose      |        |2000-05-19|M     |4000  |
|Robert   |          |Williams|1978-09-05|M     |4000  |
|Maria    |Anne      |Jones   |1967-12-01|F     |4000  |
|Jen      |Mary      |Brown   |1980-02-17|F     |-1    |
+---------+----------+--------+----------+------+------+
'''

#Change Data Type of existing column
df.printSchema()
'''
root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: long (nullable = true)
'''
#Now I am changing salary string datatype to integer
df.withColumn("salary",df.salary.cast("int")).printSchema()
'''
root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)
 '''


+---------+----------+--------+----------+------+------+
|firstname|middlename|lastname|dob       |gender|salary|
+---------+----------+--------+----------+------+------+
|James    |          |Smith   |1991-04-01|M     |3000  |
|Michael  |Rose      |        |2000-05-19|M     |4000  |
|Robert   |          |Williams|1978-09-05|M     |4000  |
|Maria    |Anne      |Jones   |1967-12-01|F     |4000  |
|Jen      |Mary      |Brown   |1980-02-17|F     |-1    |
+---------+----------+--------+----------+------+------+

root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: long (nullable = true)

root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)



In [24]:
# Update value of an existing column:
from pyspark.sql.functions import col
df.withColumn("salary",col("salary") * 100).show(truncate=False)
'''
+---------+----------+--------+----------+------+------+
|firstname|middlename|lastname|dob       |gender|salary|
+---------+----------+--------+----------+------+------+
|James    |          |Smith   |1991-04-01|M     |300000|
|Michael  |Rose      |        |2000-05-19|M     |400000|
|Robert   |          |Williams|1978-09-05|M     |400000|
|Maria    |Anne      |Jones   |1967-12-01|F     |400000|
|Jen      |Mary      |Brown   |1980-02-17|F     |-100  |
+---------+----------+--------+----------+------+------+
'''

+---------+----------+--------+----------+------+------+
|firstname|middlename|lastname|dob       |gender|salary|
+---------+----------+--------+----------+------+------+
|James    |          |Smith   |1991-04-01|M     |300000|
|Michael  |Rose      |        |2000-05-19|M     |400000|
|Robert   |          |Williams|1978-09-05|M     |400000|
|Maria    |Anne      |Jones   |1967-12-01|F     |400000|
|Jen      |Mary      |Brown   |1980-02-17|F     |-100  |
+---------+----------+--------+----------+------+------+



In [26]:
# Create a column from an existing
df.withColumn("CopiedColumn",col("salary")* +1).show(truncate=False)
'''
+---------+----------+--------+----------+------+------+------------+
|firstname|middlename|lastname|dob       |gender|salary|CopiedColumn|
+---------+----------+--------+----------+------+------+------------+
|James    |          |Smith   |1991-04-01|M     |3000  |3000        |
|Michael  |Rose      |        |2000-05-19|M     |4000  |4000        |
|Robert   |          |Williams|1978-09-05|M     |4000  |4000        |
|Maria    |Anne      |Jones   |1967-12-01|F     |4000  |4000        |
|Jen      |Mary      |Brown   |1980-02-17|F     |-1    |-1          |
+---------+----------+--------+----------+------+------+------------+
'''


+---------+----------+--------+----------+------+------+------------+
|firstname|middlename|lastname|dob       |gender|salary|CopiedColumn|
+---------+----------+--------+----------+------+------+------------+
|James    |          |Smith   |1991-04-01|M     |3000  |3000        |
|Michael  |Rose      |        |2000-05-19|M     |4000  |4000        |
|Robert   |          |Williams|1978-09-05|M     |4000  |4000        |
|Maria    |Anne      |Jones   |1967-12-01|F     |4000  |4000        |
|Jen      |Mary      |Brown   |1980-02-17|F     |-1    |-1          |
+---------+----------+--------+----------+------+------+------------+



In [28]:
# Add new column
from pyspark.sql.functions import lit
df.withColumn("Country", lit("USA")).show(truncate=False)
'''
+---------+----------+--------+----------+------+------+-------+
|firstname|middlename|lastname|dob       |gender|salary|Country|
+---------+----------+--------+----------+------+------+-------+
|James    |          |Smith   |1991-04-01|M     |3000  |USA    |
|Michael  |Rose      |        |2000-05-19|M     |4000  |USA    |
|Robert   |          |Williams|1978-09-05|M     |4000  |USA    |
|Maria    |Anne      |Jones   |1967-12-01|F     |4000  |USA    |
|Jen      |Mary      |Brown   |1980-02-17|F     |-1    |USA    |
+---------+----------+--------+----------+------+------+-------+
'''


+---------+----------+--------+----------+------+------+-------+
|firstname|middlename|lastname|dob       |gender|salary|Country|
+---------+----------+--------+----------+------+------+-------+
|James    |          |Smith   |1991-04-01|M     |3000  |USA    |
|Michael  |Rose      |        |2000-05-19|M     |4000  |USA    |
|Robert   |          |Williams|1978-09-05|M     |4000  |USA    |
|Maria    |Anne      |Jones   |1967-12-01|F     |4000  |USA    |
|Jen      |Mary      |Brown   |1980-02-17|F     |-1    |USA    |
+---------+----------+--------+----------+------+------+-------+



In [29]:
#Another Example:
df.withColumn("Country",lit("USA")) \
  .withColumn("TestColumn",lit("Test")).show(truncate=False)

'''
+---------+----------+--------+----------+------+------+-------+----------+
|firstname|middlename|lastname|dob       |gender|salary|Country|TestColumn|
+---------+----------+--------+----------+------+------+-------+----------+
|James    |          |Smith   |1991-04-01|M     |3000  |USA    |Test      |
|Michael  |Rose      |        |2000-05-19|M     |4000  |USA    |Test      |
|Robert   |          |Williams|1978-09-05|M     |4000  |USA    |Test      |
|Maria    |Anne      |Jones   |1967-12-01|F     |4000  |USA    |Test      |
|Jen      |Mary      |Brown   |1980-02-17|F     |-1    |USA    |Test      |
+---------+----------+--------+----------+------+------+-------+----------+
'''

+---------+----------+--------+----------+------+------+-------+----------+
|firstname|middlename|lastname|dob       |gender|salary|Country|TestColumn|
+---------+----------+--------+----------+------+------+-------+----------+
|James    |          |Smith   |1991-04-01|M     |3000  |USA    |Test      |
|Michael  |Rose      |        |2000-05-19|M     |4000  |USA    |Test      |
|Robert   |          |Williams|1978-09-05|M     |4000  |USA    |Test      |
|Maria    |Anne      |Jones   |1967-12-01|F     |4000  |USA    |Test      |
|Jen      |Mary      |Brown   |1980-02-17|F     |-1    |USA    |Test      |
+---------+----------+--------+----------+------+------+-------+----------+



In [31]:
# Rename Column Name:
df.printSchema()
'''
root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: long (nullable = true)
 '''
 # Now I want to rename gender column to sex
df.withColumnRenamed("gender","sex").printSchema()
'''
root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- salary: long (nullable = true)
'''

root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: long (nullable = true)

root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- salary: long (nullable = true)



In [33]:
# Drop Column
df.printSchema()
'''
root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: long (nullable = true)
'''
# I want drop gender column from the above
df.drop("gender").printSchema()
'''
root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- salary: long (nullable = true)
 '''

root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: long (nullable = true)

root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- salary: long (nullable = true)



In [5]:
from pyarrow import StructArray
# PySpark-withColumnRenamed()
from pyspark.sql.types import *
data = [(('James','','Smith'),'1991-04-01','M',3000),
  (('Michael','Rose',''),'2000-05-19','M',4000),
  (('Robert','','Williams'),'1978-09-05','M',4000),
  (('Maria','Anne','Jones'),'1967-12-01','F',4000),
  (('Jen','Mary','Brown'),'1980-02-17','F',-1)
]
schema = StructType([
    StructField("name",StructType([
        StructField("firstname",StringType(),True),
        StructField("middlename",StringType(),True),
        StructField("lastname",StringType(),True)
    ]),True),
    StructField("dob",StringType(),True),
    StructField("gender",StringType(),True),
    StructField("salary",IntegerType(),True)

])

df = spark_session.createDataFrame(data,schema)
df.show(truncate=False)
'''
+--------------------+----------+------+------+
|name                |dob       |gender|salary|
+--------------------+----------+------+------+
|{James, , Smith}    |1991-04-01|M     |3000  |
|{Michael, Rose, }   |2000-05-19|M     |4000  |
|{Robert, , Williams}|1978-09-05|M     |4000  |
|{Maria, Anne, Jones}|1967-12-01|F     |4000  |
|{Jen, Mary, Brown}  |1980-02-17|F     |-1    |
+--------------------+----------+------+------+
'''


+--------------------+----------+------+------+
|name                |dob       |gender|salary|
+--------------------+----------+------+------+
|{James, , Smith}    |1991-04-01|M     |3000  |
|{Michael, Rose, }   |2000-05-19|M     |4000  |
|{Robert, , Williams}|1978-09-05|M     |4000  |
|{Maria, Anne, Jones}|1967-12-01|F     |4000  |
|{Jen, Mary, Brown}  |1980-02-17|F     |-1    |
+--------------------+----------+------+------+



In [7]:
# Renaming DataFrame Column name
# To rename multiple column
df.printSchema()
'''
root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)
'''
df.withColumnRenamed("dob","dateOfBirth") \
  .withColumnRenamed("gender","sex").printSchema()
'''
root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- dateOfBirth: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- salary: integer (nullable = true)
 '''

root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)

root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- dateOfBirth: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- salary: integer (nullable = true)



In [11]:
# Using PySpark-StructType- To rename a nested column in DataFrame
from pyspark.sql.functions import col
schema2 = StructType([
    StructField("fname",StringType(),True),
    StructField("middlename",StringType(),True),
    StructField("lname",StringType(),True),
])

df.select(col("name").cast(schema2), \
          col("dob"),col("gender"),col("salary")).printSchema()
'''
root
 |-- name: struct (nullable = true)
 |    |-- fname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)
 '''

root
 |-- name: struct (nullable = true)
 |    |-- fname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)



In [12]:
from os import truncate
# Using Select- To rename nested elements
from pyspark.sql.functions import *
df.select(col("name.firstname").alias("fname"), \
          col("name.middlename").alias("mname"), \
          col("name.lastname").alias("lname"), \
          col("dob"),col("gender"),col("salary")).show(truncate=False)
'''
+-------+-----+--------+----------+------+------+
|fname  |mname|lname   |dob       |gender|salary|
+-------+-----+--------+----------+------+------+
|James  |     |Smith   |1991-04-01|M     |3000  |
|Michael|Rose |        |2000-05-19|M     |4000  |
|Robert |     |Williams|1978-09-05|M     |4000  |
|Maria  |Anne |Jones   |1967-12-01|F     |4000  |
|Jen    |Mary |Brown   |1980-02-17|F     |-1    |
+-------+-----+--------+----------+------+------+
'''

+-------+-----+--------+----------+------+------+
|fname  |mname|lname   |dob       |gender|salary|
+-------+-----+--------+----------+------+------+
|James  |     |Smith   |1991-04-01|M     |3000  |
|Michael|Rose |        |2000-05-19|M     |4000  |
|Robert |     |Williams|1978-09-05|M     |4000  |
|Maria  |Anne |Jones   |1967-12-01|F     |4000  |
|Jen    |Mary |Brown   |1980-02-17|F     |-1    |
+-------+-----+--------+----------+------+------+



In [13]:
# Using PySpark DataFrame withColumn- To rename nested columns
from pyspark.sql.functions import *
df.withColumn("fname", col("name.firstname")) \
  .withColumn("mname", col("name.middlename")) \
  .withColumn("lanme", col("name.lastname")) \
  .drop("name") \
  .printSchema()
'''
root
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- fname: string (nullable = true)
 |-- mname: string (nullable = true)
 |-- lanme: string (nullable = true)
'''

root
 |-- dob: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- fname: string (nullable = true)
 |-- mname: string (nullable = true)
 |-- lanme: string (nullable = true)



In [14]:
# Using toDf() – To change all columns in a PySpark DataFrame
newColumns = ["newCol1","newCol2","newCol3","newCol4"]
df.toDF(*newColumns).printSchema()
'''
root
 |-- newCol1: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- newCol2: string (nullable = true)
 |-- newCol3: string (nullable = true)
 |-- newCol4: integer (nullable = true)
 '''

root
 |-- newCol1: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- newCol2: string (nullable = true)
 |-- newCol3: string (nullable = true)
 |-- newCol4: integer (nullable = true)



In [15]:
# PySpark-Filter()
data =[
(("James","","Smith"),["Java","Scala","C++"],"OH","M"),
(("Anna","Rose",""),["Spark","Java","C++"],"NY","F"),
(("Julia","","Williams"),["CSharp","VB"],"OH","F"),
(("Maria","Anne","Jones"),["CSharp","VB"],"NY","M"),
(("Jen","Mary","Brown"),["CSharp","VB"],"NY","M"),
(("Mike","Mary","Williams"),["Python","VB"],"OH","M")
]

schema = StructType([
     StructField('name', StructType([
        StructField('firstname',StringType(),True),
        StructField('middlename',StringType(),True),
         StructField('lastname',StringType(),True)
])),
     StructField('languages',ArrayType(StringType()),True),
     StructField('state',StringType(),True),
     StructField('gender',StringType(),True)
])

df = spark_session.createDataFrame(data,schema)
df.printSchema()
'''
root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- state: string (nullable = true)
 |-- gender: string (nullable = true)
 '''
df.show(truncate=False)
'''
+----------------------+------------------+-----+------+
|name                  |languages         |state|gender|
+----------------------+------------------+-----+------+
|{James, , Smith}      |[Java, Scala, C++]|OH   |M     |
|{Anna, Rose, }        |[Spark, Java, C++]|NY   |F     |
|{Julia, , Williams}   |[CSharp, VB]      |OH   |F     |
|{Maria, Anne, Jones}  |[CSharp, VB]      |NY   |M     |
|{Jen, Mary, Brown}    |[CSharp, VB]      |NY   |M     |
|{Mike, Mary, Williams}|[Python, VB]      |OH   |M     |
+----------------------+------------------+-----+------+
'''

root
 |-- name: struct (nullable = true)
 |    |-- firstname: string (nullable = true)
 |    |-- middlename: string (nullable = true)
 |    |-- lastname: string (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- state: string (nullable = true)
 |-- gender: string (nullable = true)

+----------------------+------------------+-----+------+
|name                  |languages         |state|gender|
+----------------------+------------------+-----+------+
|{James, , Smith}      |[Java, Scala, C++]|OH   |M     |
|{Anna, Rose, }        |[Spark, Java, C++]|NY   |F     |
|{Julia, , Williams}   |[CSharp, VB]      |OH   |F     |
|{Maria, Anne, Jones}  |[CSharp, VB]      |NY   |M     |
|{Jen, Mary, Brown}    |[CSharp, VB]      |NY   |M     |
|{Mike, Mary, Williams}|[Python, VB]      |OH   |M     |
+----------------------+------------------+-----+------+



In [16]:
# Dataframe filter() with column condition:
df.filter(df.state == 'OH').show(truncate=False)
'''
+----------------------+------------------+-----+------+
|name                  |languages         |state|gender|
+----------------------+------------------+-----+------+
|{James, , Smith}      |[Java, Scala, C++]|OH   |M     |
|{Julia, , Williams}   |[CSharp, VB]      |OH   |F     |
|{Mike, Mary, Williams}|[Python, VB]      |OH   |M     |
+----------------------+------------------+-----+------+
'''

+----------------------+------------------+-----+------+
|name                  |languages         |state|gender|
+----------------------+------------------+-----+------+
|{James, , Smith}      |[Java, Scala, C++]|OH   |M     |
|{Julia, , Williams}   |[CSharp, VB]      |OH   |F     |
|{Mike, Mary, Williams}|[Python, VB]      |OH   |M     |
+----------------------+------------------+-----+------+



In [17]:
# not equals condition
df.filter(df.state!='OH').show(truncate=False)
df.filter(~(df.state == 'OH')).show(truncate=False)

'''
+--------------------+------------------+-----+------+
|name                |languages         |state|gender|
+--------------------+------------------+-----+------+
|{Anna, Rose, }      |[Spark, Java, C++]|NY   |F     |
|{Maria, Anne, Jones}|[CSharp, VB]      |NY   |M     |
|{Jen, Mary, Brown}  |[CSharp, VB]      |NY   |M     |
+--------------------+------------------+-----+------+
'''

+--------------------+------------------+-----+------+
|name                |languages         |state|gender|
+--------------------+------------------+-----+------+
|{Anna, Rose, }      |[Spark, Java, C++]|NY   |F     |
|{Maria, Anne, Jones}|[CSharp, VB]      |NY   |M     |
|{Jen, Mary, Brown}  |[CSharp, VB]      |NY   |M     |
+--------------------+------------------+-----+------+

+--------------------+------------------+-----+------+
|name                |languages         |state|gender|
+--------------------+------------------+-----+------+
|{Anna, Rose, }      |[Spark, Java, C++]|NY   |F     |
|{Maria, Anne, Jones}|[CSharp, VB]      |NY   |M     |
|{Jen, Mary, Brown}  |[CSharp, VB]      |NY   |M     |
+--------------------+------------------+-----+------+

